# Experiment Tracking & Reproducibility Cheatsheet

> Tools and patterns for tracking ML experiments, managing artifacts, and ensuring reproducibility.

---
## MLflow

### Setup
```bash
pip install mlflow
mlflow ui --port 5000   # Launch tracking UI
```

In [ ]:
# MLflow Experiment Tracking
import mlflow

# Set experiment
mlflow.set_experiment("fraud-detection")

# Basic tracking
with mlflow.start_run(run_name="random-forest-v1"):
    # Log parameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("learning_rate", 0.01)

    # Log metrics
    mlflow.log_metric("accuracy", 0.95)
    mlflow.log_metric("f1_score", 0.93)
    mlflow.log_metric("auc_roc", 0.98)

    # Log step metrics (e.g., per epoch)
    for epoch in range(5):
        mlflow.log_metric("train_loss", 1.0 / (epoch + 1), step=epoch)

    # Log artifacts
    # mlflow.log_artifact("confusion_matrix.png")
    # mlflow.sklearn.log_model(model, "model")

    # Set tags
    mlflow.set_tag("model_type", "random_forest")
    mlflow.set_tag("dataset_version", "v2.1")

print("MLflow run logged successfully")

In [ ]:
# MLflow Auto-logging (framework-specific)
import mlflow

# Auto-log for common frameworks:
# mlflow.sklearn.autolog()     # scikit-learn
# mlflow.pytorch.autolog()     # PyTorch
# mlflow.tensorflow.autolog()  # TensorFlow
# mlflow.xgboost.autolog()     # XGBoost
# mlflow.lightgbm.autolog()    # LightGBM

# Example with sklearn:
# mlflow.sklearn.autolog()
# with mlflow.start_run():
#     model = RandomForestClassifier(n_estimators=100)
#     model.fit(X_train, y_train)  # auto-logs params, metrics, model

print("MLflow auto-logging: automatically captures params, metrics, and model artifacts")

### MLflow CLI

```bash
# List experiments
mlflow experiments search

# Search runs
mlflow runs list --experiment-id 1

# Compare metrics
mlflow runs list --experiment-id 1 -o table \
  --filter "metrics.accuracy > 0.9"

# Register model
mlflow models register --name "fraud-model" \
  --source "runs:/<run-id>/model"

# Serve model
mlflow models serve -m "models:/fraud-model/Production" -p 5001
```

### MLflow Projects

```yaml
# MLproject file
name: fraud-detection
conda_env: conda.yml
entry_points:
  main:
    parameters:
      learning_rate: {type: float, default: 0.01}
      epochs: {type: int, default: 10}
    command: "python train.py --lr {learning_rate} --epochs {epochs}"
  evaluate:
    parameters:
      model_uri: {type: str}
    command: "python evaluate.py --model-uri {model_uri}"
```

```bash
# Run project
mlflow run . -P learning_rate=0.001 -P epochs=20

# Run from git
mlflow run https://github.com/user/project.git -P epochs=10
```

---
## Weights & Biases (W&B)

### Setup
```bash
pip install wandb
wandb login
```

In [ ]:
# Weights & Biases Tracking (reference code)
import wandb

# Initialize run
# run = wandb.init(
#     project="fraud-detection",
#     name="xgboost-v2",
#     config={
#         "learning_rate": 0.01,
#         "epochs": 10,
#         "batch_size": 64,
#         "architecture": "xgboost",
#     },
#     tags=["baseline", "xgboost"]
# )

# Log metrics during training
# for epoch in range(config.epochs):
#     train_loss = train_one_epoch(model, train_loader)
#     val_loss, val_acc = evaluate(model, val_loader)
#     wandb.log({
#         "epoch": epoch,
#         "train_loss": train_loss,
#         "val_loss": val_loss,
#         "val_accuracy": val_acc,
#     })

# Log artifacts
# artifact = wandb.Artifact("model", type="model")
# artifact.add_file("model.pkl")
# run.log_artifact(artifact)

# run.finish()
print("W&B tracking pattern defined")

In [ ]:
# W&B Hyperparameter Sweeps (reference code)
import wandb

sweep_config = {
    "method": "bayes",         # bayes, grid, random
    "metric": {
        "name": "val_accuracy",
        "goal": "maximize"
    },
    "parameters": {
        "learning_rate": {
            "min": 0.0001,
            "max": 0.1,
            "distribution": "log_uniform_values"
        },
        "batch_size": {
            "values": [16, 32, 64, 128]
        },
        "epochs": {
            "value": 10
        }
    }
}

# sweep_id = wandb.sweep(sweep_config, project="fraud-detection")
# wandb.agent(sweep_id, function=train, count=20)
print("W&B sweep config:", sweep_config["method"], "optimization")

---
## DVC (Data Version Control)

### Setup & Data Versioning
```bash
pip install dvc dvc-s3
dvc init

# Track data
dvc add data/training_data.csv
git add data/training_data.csv.dvc data/.gitignore
git commit -m "Add training data v1"

# Remote storage
dvc remote add -d myremote s3://my-bucket/dvc-store
dvc push
dvc pull
```

### DVC Experiments
```bash
# Run experiment
dvc exp run -S train.lr=0.001 -S train.epochs=20

# Compare experiments
dvc exp show --sort-by metrics.accuracy --sort-order desc

# Apply best experiment to workspace
dvc exp apply <exp-name>

# Push experiment to Git
dvc exp push origin <exp-name>
```

### DVC Pipeline
```yaml
# dvc.yaml
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - data/raw/
      - src/prepare.py
    outs:
      - data/processed/
  train:
    cmd: python src/train.py
    deps:
      - data/processed/
      - src/train.py
    params:
      - train.lr
      - train.epochs
    outs:
      - models/model.pkl
    metrics:
      - metrics.json:
          cache: false
```

```bash
dvc repro              # Reproduce pipeline
dvc dag                # Show pipeline DAG
dvc metrics show       # Show metrics
dvc metrics diff       # Compare with previous
```

---
## Reproducibility Checklist

| Category | Action | Tool |
|----------|--------|------|
| **Code** | Version control all code | Git |
| **Data** | Version control datasets | DVC, LakeFS |
| **Environment** | Pin all dependencies | pip freeze, conda export |
| **Config** | Track all hyperparameters | MLflow, W&B, Hydra |
| **Random seeds** | Set seeds for all libraries | PyTorch, NumPy, Python |
| **Hardware** | Log hardware configuration | nvidia-smi, platform |
| **Results** | Log all metrics and artifacts | MLflow, W&B |

In [ ]:
# Reproducibility: Setting Random Seeds
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For full determinism (may reduce performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f"Seeds set for: random, numpy, torch (seed=42)")
print(f"Random: {random.random():.6f}")
print(f"Numpy:  {np.random.random():.6f}")

## Tool Comparison

| Feature | MLflow | W&B | DVC |
|---------|--------|-----|-----|
| **Tracking** | Yes | Yes | Yes (via params/metrics) |
| **Visualization** | Basic UI | Rich dashboards | CLI-based |
| **Model Registry** | Yes | Yes | No (use MLflow) |
| **Data Versioning** | No | Artifacts | Yes (core feature) |
| **Hyperparameter Tuning** | No (use Optuna) | Sweeps | Experiments |
| **Collaboration** | Self-hosted / Managed | Cloud-first | Git-based |
| **Cost** | Free (OSS) | Free tier + paid | Free (OSS) |
| **Best For** | Full MLOps, model serving | Team collaboration, vis | Data + pipeline versioning |

## Interview Scenarios

**Q: How do you ensure reproducibility in ML experiments?**
> Five pillars: (1) version code with Git, (2) version data with DVC, (3) pin all dependencies (pip freeze), (4) log all hyperparameters and metrics (MLflow/W&B), (5) set random seeds across all libraries. Additionally, log hardware config and use Docker for environment reproducibility.

**Q: How would you set up experiment tracking for a team of ML engineers?**
> Use MLflow with a shared tracking server (PostgreSQL backend + S3 artifact store) or W&B (cloud-hosted). Key requirements: shared experiment dashboard, model registry for promotion workflow, auto-logging integration, CI/CD integration for automated experiment runs, and access controls.